<a href="https://colab.research.google.com/github/daviseemann/turbofan-rul-prediction-cmapss/blob/production/notebooks/autoencoder-0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
drive.mount('/content/drive/', )
import os
os.chdir('/content/drive/MyDrive/Data science studies/Aprendizado-de-maquina-UFSC/final-project/data')

In [ ]:
# Caminhos dos arquivos
train_path = "./6.turbofan rul/train_FD001.txt"
test_path = "./6.turbofan rul/test_FD001.txt"
rul_path = "./6.turbofan rul/RUL_FD001.txt"

# Nomes das colunas (de acordo com a documentação original do C-MAPSS)
column_names = ['engine_id', 'cycle'] + \
               [f'op_setting_{i}' for i in range(1, 4)] + \
               [f'sensor_measurement_{i}' for i in range(1, 22)]

# Importando os arquivos (espaço em branco como delimitador)
df_train = pd.read_csv(train_path, sep='\s+', header=None, names=column_names)
df_test = pd.read_csv(test_path, sep='\s+', header=None, names=column_names)
df_rul = pd.read_csv(rul_path, sep='\s+', header=None, names=['RUL'])

In [ ]:
df = df_train.copy()

In [ ]:
SELECTED_SENSORS = [2, 3, 4, 7, 8, 9, 11, 12, 13, 14, 15, 17, 20, 21]
sensor_cols = [f'sensor_measurement_{i}' for i in SELECTED_SENSORS]
features = df[['engine_id', 'cycle'] + sensor_cols].copy()

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(-1, 1))
features[sensor_cols] = scaler.fit_transform(features[sensor_cols])

In [ ]:
EARLY_RUL = 129

# Cálculo da vida útil
max_cycle = features.groupby('engine_id')['cycle'].max().reset_index()
max_cycle.columns = ['engine_id', 'max_cycle']
features = features.merge(max_cycle, on='engine_id')
features['RUL'] = features['max_cycle'] - features['cycle']
features['RUL'] = features['RUL'].clip(upper=EARLY_RUL)
features.drop(columns=['max_cycle'], inplace=True)

In [ ]:
def generate_windows(df, sequence_length=30, drop_cols=['engine_id', 'cycle', 'RUL'], label_col='RUL'):
    """
    Gera janelas temporais para o MLP.
    Retorna X (features) e y (RUL no final da janela).
    """
    X, y = [], []
    engine_ids = df['engine_id'].unique()

    for eid in engine_ids:
        engine_data = df[df['engine_id'] == eid].reset_index(drop=True)
        for i in range(len(engine_data) - sequence_length + 1):
            window = engine_data.iloc[i:i + sequence_length]
            features = window.drop(columns=drop_cols).values.flatten()
            label = window[label_col].values[-1]
            X.append(features)
            y.append(label)

    return np.array(X), np.array(y)


In [ ]:
X_train, y_train = generate_windows(features, window_size=24, stride=1)
print("Formato de entrada X:", X_train.shape)
print("Formato dos labels y:", y_train.shape)

In [ ]:
plt.plot(y_train[:200])
plt.title("RUL das primeiras 200 amostras")
plt.xlabel("Amostra")
plt.ylabel("RUL")
plt.grid()
plt.show()